# CS604 Colab launcher

Runtime: **GPU**. Images stay on Colab disk (`/content/data`). Checkpoints and logs go to the shared Drive folder **CS604_RUNS**.

The GitHub repo is private, so the clone cell will ask for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it.

**Drive:** everyone must have `CS604_RUNS` under *My Drive* (create it there, or open the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR) → Organize → **Add shortcut to Drive** → My Drive, name it `CS604_RUNS`). A Drive *link* is not a file path Colab can write to.

Change `METHOD`, `TEAM`, and `BRANCH` below before you run.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "wenqi"  # code lives here until we merge to main
METHOD = "linear_probe"  # linear_probe | full_ft | lora_r8 | bitfit | last_blocks | prototype_probe | prototype_lora_r8
TEAM = "wenqi"
PROTO_PT = f"{DRIVE_OUT}/prototypes.pt"

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes (once per backbone)

Van's ~43.9% zero-shot head. Run **once**, then reuse `PROTO_PT` for `prototype_probe` / `prototype_lora_r8`. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/prototype_probe.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## Review settings (table)

Loads `configs/{METHOD}.yaml` (merged with `base.yaml`) so you can check every parameter before spending a GPU hour.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

methods = ["full_ft", "linear_probe", "lora_r8", "bitfit", "last_blocks", "prototype_probe", "prototype_lora_r8", "adapter", "adaptformer"]
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
keys = []
for d in merged.values():
    for k in d:
        if k not in keys:
            keys.append(k)

compare = pd.DataFrame(
    {name: [merged[name].get(k, "") for k in keys] for name in methods},
    index=keys,
)
compare.index.name = "parameter"
print("Resolved parameters per method (base.yaml already merged in):")
display(compare)

chosen = load_config(f"configs/{METHOD}.yaml")
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
resolved = pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"])
print(f"\nThis run (METHOD={METHOD!r}, plus Colab data_root / output_dir):")
display(resolved)

## 5. Train (checkpoints on Drive)

In [ ]:
!python -m src.train --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output_dir {DRIVE_OUT}

## 6. Predict

Set `CKPT` to the `best.pt` (or `last.pt`) from the run folder printed above.

In [ ]:
CKPT = f"{DRIVE_OUT}/REPLACE_WITH_RUN_DIR/best.pt"

!python -m src.predict --config configs/{METHOD}.yaml --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}
!python -m src.predict --config configs/{METHOD}.yaml --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}